<a href="https://colab.research.google.com/github/hydroz3/flyrank-ml-intern/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hydroz3/flyrank-ml-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

'''
1. High impressions and low clicks (CTR)
# gsc_clicks / gsc_impressions
High impressions but relatively few clicks.
The page is shown frequently in search results, but its click-through rate is relatively low.
This suggests strong visibility but weak click response.

2. Session Engagement Rate (ENGAGE_RATE)
# ga4_engaged_sessions / ga4_sessions
The page receives sessions, but relatively few of them are engaged sessions.
This suggests that users are reaching the page, but the content may not be keeping them engaged.

3. High CTR but low sessions (LOW_TRAFFIC_GOOD_CTR)
# CTR = gsc_clicks / gsc_impressions
# then compare CTR with ga4_sessions
The page has a relatively good click-through rate, but still receives few sessions.
Users tend to click when the page is shown, but the overall traffic volume remains limited.

Rule 1: Are people clicking?
Rule 2: Are visitors engaging?
Rule 3: Is good click efficiency translating into enough traffic?
'''

'\n1. High impressions and low clicks (CTR)\n# gsc_clicks / gsc_impressions\nHigh impressions but relatively few clicks.\nThe page is shown frequently in search results, but its click-through rate is relatively low.\nThis suggests strong visibility but weak click response.\n\n2. Session Engagement Rate (ENGAGE_RATE)\n# ga4_engaged_sessions / ga4_sessions\nThe page receives sessions, but relatively few of them are engaged sessions.\nThis suggests that users are reaching the page, but the content may not be keeping them engaged.\n\n3. High CTR but low sessions (LOW_TRAFFIC_GOOD_CTR) \n# CTR = gsc_clicks / gsc_impressions\n# then compare CTR with ga4_sessions\nThe page has a relatively good click-through rate, but still receives few sessions.\nUsers tend to click when the page is shown, but the overall traffic volume remains limited.\n\nRule 1: Are people clicking?\nRule 2: Are visitors engaging?\nRule 3: Is good click efficiency translating into enough traffic?\n'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)



Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [3]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [4]:
preview = con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-05/*.parquet'
)
LIMIT 5
""").df()

preview

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-05-01,client_3ffa76342f366962,content_4eb35f25cf28b104,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-05
1,2026-05-01,client_3ffa76342f366962,content_5e4bf89ca5706ca2,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-05
2,2026-05-01,client_3ffa76342f366962,content_2dd03169b60f7a43,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-05
3,2026-05-01,client_3ffa76342f366962,content_93ea5d35d707f873,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-05
4,2026-05-01,client_3ffa76342f366962,content_c000c4c9e218328b,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-05


In [5]:
df = con.sql(f"""
SELECT
    *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-05/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
print(df.shape)
df.head()

(11687376, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-05-01,client_3ffa76342f366962,content_4eb35f25cf28b104,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-05
1,2026-05-01,client_3ffa76342f366962,content_5e4bf89ca5706ca2,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-05
2,2026-05-01,client_3ffa76342f366962,content_2dd03169b60f7a43,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-05
3,2026-05-01,client_3ffa76342f366962,content_93ea5d35d707f873,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-05
4,2026-05-01,client_3ffa76342f366962,content_c000c4c9e218328b,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-05


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11687376 entries, 0 to 11687375
Data columns (total 31 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   report_date               datetime64[us]
 1   client_hash_id            object        
 2   content_hash_id           object        
 3   client_has_gsc            bool          
 4   client_has_ga4            bool          
 5   gsc_data_available        bool          
 6   ga4_data_available        boolean       
 7   gsc_impressions           int64         
 8   gsc_clicks                int64         
 9   gsc_sum_position          int64         
 10  gsc_avg_position          float64       
 11  ga4_pageviews             Int64         
 12  ga4_sessions              Int64         
 13  ga4_users                 Int64         
 14  ga4_engaged_sessions      Int64         
 15  ga4_total_engagement_sec  Int64         
 16  sessions_organic          Int64         
 17  sessio

In [8]:
import numpy as np

df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

In [9]:
df[
    ["gsc_impressions", "gsc_clicks", "ctr"]
].head()

,gsc_impressions,gsc_clicks,ctr
0,0,0,NaN
1,0,0,NaN
2,0,0,NaN
3,0,0,NaN
4,0,0,NaN


In [10]:
df["engagement_rate"] = (
    df["ga4_engaged_sessions"] / df["ga4_sessions"]
).where(df["ga4_sessions"].fillna(0) > 0)

In [11]:
df[
    [
        "ga4_sessions",
        "ga4_engaged_sessions",
        "engagement_rate"
    ]
].head()

,ga4_sessions,ga4_engaged_sessions,engagement_rate
0,0,0,<NA>
1,0,0,<NA>
2,0,0,<NA>
3,0,0,<NA>
4,0,0,<NA>


In [12]:
df.loc[
    (df["gsc_impressions"] > 0)
    & (df["gsc_clicks"] > 0),
    ["gsc_impressions", "gsc_clicks", "ctr"]
].head()

,gsc_impressions,gsc_clicks,ctr
853,2,1,0.500000
2595,1,1,1.000000
6105,1,1,1.000000
9213,39,1,0.025641
15315,344,3,0.008721


In [13]:
df.loc[
    (df["ga4_sessions"].fillna(0) > 0)
    & (df["ga4_engaged_sessions"].fillna(0) > 0),
    ["ga4_sessions", "ga4_engaged_sessions", "engagement_rate"]
].head()

,ga4_sessions,ga4_engaged_sessions,engagement_rate
15444,3,1,0.333333
15478,1,1,1.0
15516,2,1,0.5
15587,1,1,1.0
15620,1,1,1.0


In [14]:
high_impressions_threshold = (
    df.loc[df["gsc_impressions"] > 0, "gsc_impressions"]
    .quantile(0.75)
)

low_ctr_threshold = (
    df.loc[df["ctr"] > 0, "ctr"]
    .quantile(0.25)
)

high_ctr_threshold = (
    df.loc[df["ctr"] > 0, "ctr"]
    .quantile(0.75)
)

low_engagement_threshold = (
    df.loc[df["engagement_rate"] > 0, "engagement_rate"]
    .quantile(0.25)
)

low_sessions_threshold = (
    df.loc[df["ga4_sessions"] > 0, "ga4_sessions"]
    .quantile(0.25)
)

In [15]:
print("High impressions threshold:", high_impressions_threshold)
print("Low CTR threshold:", low_ctr_threshold)
print("High CTR threshold:", high_ctr_threshold)
print("Low engagement threshold:", low_engagement_threshold)
print("Low sessions threshold:", low_sessions_threshold)

High impressions threshold: 39.0
Low CTR threshold: 0.005319148936170213
High CTR threshold: 0.02564102564102564
Low engagement threshold: 0.2
Low sessions threshold: 1.0


In [16]:
df["HIGH_VIS_LOW_CLICKS"] = (
    (df["gsc_impressions"] >= high_impressions_threshold)
    &
    (df["ctr"] <= low_ctr_threshold)
).astype(int)

In [17]:
df["LOW_ENGAGEMENT_RATE"] = (
    (df["ga4_sessions"].fillna(0) > 0)
    &
    (df["engagement_rate"].fillna(1) <= low_engagement_threshold)
).astype(int)

In [18]:
df["LOW_TRAFFIC_GOOD_CTR"] = (
    (df["gsc_impressions"].fillna(0) >= high_impressions_threshold)
    &
    (df["ctr"].fillna(0) >= high_ctr_threshold)
    &
    (df["ga4_sessions"].notna())
    &
    (df["ga4_sessions"] <= low_sessions_threshold)
).astype(int)

In [19]:
rule_cols = [
    "HIGH_VIS_LOW_CLICKS",
    "LOW_ENGAGEMENT_RATE",
    "LOW_TRAFFIC_GOOD_CTR"
]

df[rule_cols].sum()

,0
HIGH_VIS_LOW_CLICKS,839141
LOW_ENGAGEMENT_RATE,692244
LOW_TRAFFIC_GOOD_CTR,5799


In [20]:
df[rule_cols].mean() * 100

,0
HIGH_VIS_LOW_CLICKS,7.179892
LOW_ENGAGEMENT_RATE,5.923006
LOW_TRAFFIC_GOOD_CTR,0.049618


In [21]:
df["baseline_action_score"] = (
    df["HIGH_VIS_LOW_CLICKS"]
    + df["LOW_ENGAGEMENT_RATE"]
    + df["LOW_TRAFFIC_GOOD_CTR"]
)

In [22]:
df["baseline_action_score"].value_counts().sort_index()

,count
baseline_action_score,
0,10370658
1,1096252
2,220466


In [23]:
def make_reason_codes(row):
    reasons = []

    if row["HIGH_VIS_LOW_CLICKS"] == 1:
        reasons.append("HIGH_VIS_LOW_CLICKS")

    if row["LOW_ENGAGEMENT_RATE"] == 1:
        reasons.append("LOW_ENGAGEMENT_RATE")

    if row["LOW_TRAFFIC_GOOD_CTR"] == 1:
        reasons.append("LOW_TRAFFIC_GOOD_CTR")

    return ", ".join(reasons)

In [24]:
df["reason_codes"] = ""

df.loc[
    df["HIGH_VIS_LOW_CLICKS"] == 1,
    "reason_codes"
] += "HIGH_VIS_LOW_CLICKS, "

df.loc[
    df["LOW_ENGAGEMENT_RATE"] == 1,
    "reason_codes"
] += "LOW_ENGAGEMENT_RATE, "

df.loc[
    df["LOW_TRAFFIC_GOOD_CTR"] == 1,
    "reason_codes"
] += "LOW_TRAFFIC_GOOD_CTR, "

df["reason_codes"] = df["reason_codes"].str.rstrip(", ")

In [25]:
df[
    [
        "baseline_action_score",
        "reason_codes"
    ]
].head(20)

,baseline_action_score,reason_codes
0,0,
1,0,
2,0,
3,0,
4,0,
5,0,
6,0,
7,0,
8,0,
9,0,


In [26]:
df["baseline_action_score"].value_counts().sort_index()

,count
baseline_action_score,
0,10370658
1,1096252
2,220466


In [30]:
df.loc[
    df["baseline_action_score"] > 0,
    ["baseline_action_score", "reason_codes"]
].head(20)

,baseline_action_score,reason_codes
579,1,LOW_ENGAGEMENT_RATE
853,1,LOW_ENGAGEMENT_RATE
2595,1,LOW_ENGAGEMENT_RATE
3354,1,HIGH_VIS_LOW_CLICKS
5191,1,LOW_ENGAGEMENT_RATE
5276,1,LOW_ENGAGEMENT_RATE
8369,1,LOW_ENGAGEMENT_RATE
9213,2,"LOW_ENGAGEMENT_RATE, LOW_TRAFFIC_GOOD_CTR"
15314,1,HIGH_VIS_LOW_CLICKS
15317,1,HIGH_VIS_LOW_CLICKS


In [31]:
ranked_queue = df.loc[
    df["baseline_action_score"] > 0,
    [
        "content_hash_id",  # replace if your identifier column has another name
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "engagement_rate",
        "baseline_action_score",
        "reason_codes"
    ]
].copy()

In [32]:
print("Full df:", df.shape)
print("Ranked candidates:", ranked_queue.shape)

Full df: (11687376, 38)
Ranked candidates: (1316718, 10)


In [33]:
ranked_queue.sort_values(
    by=[
        "baseline_action_score",
        "gsc_impressions"
    ],
    ascending=[
        False,
        False
    ],
    inplace=True
)

In [34]:
ranked_queue.reset_index(
    drop=True,
    inplace=True
)

In [35]:
ranked_queue.insert(
    0,
    "baseline_rank",
    range(1, len(ranked_queue) + 1)
)

In [41]:
output_columns = [
    "baseline_rank",
    "content_hash_id",

    "baseline_action_score",
    "reason_codes",

    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",

    "ga4_sessions",
    "ga4_engaged_sessions",
    "engagement_rate"
]

ranked_queue = ranked_queue[output_columns]

In [38]:
ranked_queue.head(20)

,baseline_rank,content_hash_id,baseline_action_score,reason_codes,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,engagement_rate
0,1,content_cedadfe5ae4845ac,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",47607,57,0.001197,5.632722,61,1,0.016393
1,2,content_21309e9a83c83653,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",46325,58,0.001252,5.071689,42,0,0.0
2,3,content_65c75874a23fca87,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",32168,2,0.000062,8.426604,2,0,0.0
3,4,content_2f567cb6ad16f6f0,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",28431,3,0.000106,9.095741,4,0,0.0
4,5,content_545bb6cc7081ded3,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",26663,84,0.003150,2.153734,43,2,0.046512
5,6,content_86c96002dd5c69aa,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",25922,62,0.002392,10.144433,78,8,0.102564
6,7,content_21309e9a83c83653,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",25806,25,0.000969,5.178602,14,1,0.071429
7,8,content_21309e9a83c83653,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",24429,38,0.001556,5.006877,27,0,0.0
8,9,content_65c75874a23fca87,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",24391,2,0.000082,8.963142,3,0,0.0
9,10,content_545bb6cc7081ded3,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",22798,85,0.003728,2.148083,56,3,0.053571


In [43]:
ranked_queue[
    [
        "baseline_rank",
        "content_hash_id",
        "baseline_action_score",
        "reason_codes",
        "gsc_impressions",
        "ctr",
        "ga4_sessions",
        "engagement_rate"
    ]
].head(20)

,baseline_rank,content_hash_id,baseline_action_score,reason_codes,gsc_impressions,ctr,ga4_sessions,engagement_rate
0,1,content_cedadfe5ae4845ac,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",47607,0.001197,61,0.016393
1,2,content_21309e9a83c83653,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",46325,0.001252,42,0.0
2,3,content_65c75874a23fca87,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",32168,0.000062,2,0.0
3,4,content_2f567cb6ad16f6f0,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",28431,0.000106,4,0.0
4,5,content_545bb6cc7081ded3,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",26663,0.003150,43,0.046512
5,6,content_86c96002dd5c69aa,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",25922,0.002392,78,0.102564
6,7,content_21309e9a83c83653,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",25806,0.000969,14,0.071429
7,8,content_21309e9a83c83653,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",24429,0.001556,27,0.0
8,9,content_65c75874a23fca87,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",24391,0.000082,3,0.0
9,10,content_545bb6cc7081ded3,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",22798,0.003728,56,0.053571


In [44]:
import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)

In [46]:
output_path = "work/outputs/baseline_action_score.csv"

ranked_queue.to_csv(
    output_path,
    index=False
)

In [47]:
ranked_queue.to_csv(
    output_path,
    columns=output_columns,
    index=False
)

In [48]:
os.path.exists(output_path)

True

In [49]:
os.path.getsize(output_path)

122276747

In [50]:
print(
    "CSV size:",
    round(os.path.getsize(output_path) / 1024 / 1024, 2),
    "MB"
)

CSV size: 116.61 MB


In [51]:
import pandas as pd

check = pd.read_csv(
    output_path,
    nrows=5
)

check

,baseline_rank,content_hash_id,baseline_action_score,reason_codes,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,engagement_rate
0,1,content_cedadfe5ae4845ac,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",47607,57,0.001197,5.632722,61,1,0.016393
1,2,content_21309e9a83c83653,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",46325,58,0.001252,5.071689,42,0,0.000000
2,3,content_65c75874a23fca87,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",32168,2,0.000062,8.426604,2,0,0.000000
3,4,content_2f567cb6ad16f6f0,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",28431,3,0.000106,9.095741,4,0,0.000000
4,5,content_545bb6cc7081ded3,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",26663,84,0.003150,2.153734,43,2,0.046512


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# precision


In [52]:
top20 = ranked_queue.head(20).copy()

top20[
    [
        "baseline_rank",
        "baseline_action_score",
        "reason_codes",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "engagement_rate"
    ]
]

,baseline_rank,baseline_action_score,reason_codes,gsc_impressions,gsc_clicks,ctr,ga4_sessions,ga4_engaged_sessions,engagement_rate
0,1,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",47607,57,0.001197,61,1,0.016393
1,2,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",46325,58,0.001252,42,0,0.0
2,3,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",32168,2,0.000062,2,0,0.0
3,4,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",28431,3,0.000106,4,0,0.0
4,5,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",26663,84,0.003150,43,2,0.046512
5,6,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",25922,62,0.002392,78,8,0.102564
6,7,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",25806,25,0.000969,14,1,0.071429
7,8,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",24429,38,0.001556,27,0,0.0
8,9,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",24391,2,0.000082,3,0,0.0
9,10,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",22798,85,0.003728,56,3,0.053571


In [53]:
top20["action"] = ""
top20["confidence_note"] = ""
top20["what_would_make_it_wrong"] = ""

In [54]:
def get_action(reason):
    if "HIGH_VIS_LOW_CLICKS" in reason:
        return "Review title/meta snippet and search intent"

    elif "LOW_ENGAGEMENT_RATE" in reason:
        return "Review page content and engagement quality"

    elif "LOW_TRAFFIC_GOOD_CTR" in reason:
        return "Investigate why good CTR is not producing more sessions"

    return "No action"

In [55]:
top20["action"] = top20["reason_codes"].apply(get_action)

In [56]:
top20["confidence_note"] = top20["baseline_action_score"].map({
    3: "High: multiple baseline signals triggered",
    2: "Medium-high: two signals triggered",
    1: "Medium: one signal triggered"
})

In [57]:
def wrong_reason(row):
    if row["gsc_impressions"] <= 5:
        return "Very small search volume may make the ratios unstable"

    if row["ga4_sessions"] <= 1:
        return "Very low session volume may make engagement metrics unreliable"

    return "Could be a false positive if the page is intentionally low-traffic or serves a niche intent"

In [58]:
top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)

In [59]:
top20[
    [
        "baseline_rank",
        "baseline_action_score",
        "reason_codes",
        "action",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,baseline_rank,baseline_action_score,reason_codes,action,confidence_note,what_would_make_it_wrong
0,1,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",Review title/meta snippet and search intent,Medium-high: two signals triggered,Could be a false positive if the page is inten...
1,2,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",Review title/meta snippet and search intent,Medium-high: two signals triggered,Could be a false positive if the page is inten...
2,3,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",Review title/meta snippet and search intent,Medium-high: two signals triggered,Could be a false positive if the page is inten...
3,4,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",Review title/meta snippet and search intent,Medium-high: two signals triggered,Could be a false positive if the page is inten...
4,5,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",Review title/meta snippet and search intent,Medium-high: two signals triggered,Could be a false positive if the page is inten...
5,6,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",Review title/meta snippet and search intent,Medium-high: two signals triggered,Could be a false positive if the page is inten...
6,7,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",Review title/meta snippet and search intent,Medium-high: two signals triggered,Could be a false positive if the page is inten...
7,8,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",Review title/meta snippet and search intent,Medium-high: two signals triggered,Could be a false positive if the page is inten...
8,9,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",Review title/meta snippet and search intent,Medium-high: two signals triggered,Could be a false positive if the page is inten...
9,10,2,"HIGH_VIS_LOW_CLICKS, LOW_ENGAGEMENT_RATE",Review title/meta snippet and search intent,Medium-high: two signals triggered,Could be a false positive if the page is inten...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

'''
Weak picks are mainly pages with very small impression or session counts, because ratios such as CTR and engagement rate become unstable at low volume.

The baseline only use May 2026 search and GA4 signals (gsc_impressions, gsc_clicks, ctr, ga4_sessions, ga4_engaged_sessions and engagement_rate). There are no product flags, future outcome windows or later-month labels were used in the score, so no obvious leakage is detected.
'''


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.